# Exercise 63 — `.apply()` vs vectorization

## Concept

`.apply()` runs a Python function over each row or value. It's flexible but **slow** — it can't use the vectorized C loops that make pandas fast. On a 10M-row frame, the gap is often 100–1000×.

```python
# SLOW — Python function called per row/value
df["squared"] = df["x"].apply(lambda v: v * v)
df["squared"] = df["x"].apply(square)

# FAST — vectorized
df["squared"] = df["x"] ** 2
```

### What to reach for instead of `.apply`

| Want                                  | Vectorized form                                          |
|---------------------------------------|----------------------------------------------------------|
| math on a column                      | Arithmetic operators on the Series                       |
| math across multiple columns          | Arithmetic operators on Series                           |
| if/else on one column                 | `np.where(cond, a, b)`                                   |
| multiple branches                     | `np.select([c1, c2, ...], [v1, v2, ...], default=...)`   |
| string ops                            | `df["name"].str.upper()`, `.str.contains(...)`, etc.     |
| datetime ops                          | `df["ts"].dt.year`, etc.                                 |
| map a value → another value           | `df["region"].map({"N": "north", "S": "south"})`         |

### When `.apply` is acceptable

- The logic genuinely can't be vectorized (calling an external service, complex parsing, regex with side-effects)
- The DataFrame is small (< ~10k rows) and clarity matters more than speed
- A prototype / exploratory analysis where you'll throw the code away

Even then, prefer `df["col"].apply(fn)` (per value) over `df.apply(fn, axis=1)` (per row) — the latter is the slowest pattern in pandas. If you must process rows together, often `.apply(fn, axis=1)` can be rewritten as separate vectorized operations on each column.

## Setup

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6],
    "customer": ["Alice Smith", "BOB DOE", "  carol  ", "Diana", "eve woods", "Frank"],
    "region":   ["N", "S", "N", "E", "S", "E"],
    "amount":   [100.0, 45.0, 300.0, 89.0, 210.0, 55.0],
    "qty":      [2, 1, 5, 3, 4, 1],
})
print(df)


## Task 1 — Vectorize an arithmetic .apply

The code below uses `.apply` with a lambda to compute `unit_price = amount / qty`. **Rewrite it** without `.apply`.

```python
# Slow original:
df["unit_price"] = df.apply(lambda row: row["amount"] / row["qty"], axis=1)
```

Return a new DataFrame with the `unit_price` column added.

```python
def add_unit_price(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

In [ ]:
import pandas as pd

def add_unit_price(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO  (one line in the body, no .apply)

result = add_unit_price(df)
assert result["unit_price"].tolist() == [50.0, 45.0, 60.0, 89/3, 52.5, 55.0]
print("ok")


## Task 2 — Vectorize string ops with `.str`

Return a new DataFrame with a `customer_clean` column that is the customer's name **stripped of whitespace and title-cased** (`"alice smith"` → `"Alice Smith"`).

Use `.str` accessors — no `.apply`.

```python
def clean_customer(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Expected `customer_clean`:
```
["Alice Smith", "Bob Doe", "Carol", "Diana", "Eve Woods", "Frank"]
```

Hint: `df["customer"].str.strip().str.title()`.

In [ ]:
import pandas as pd

def clean_customer(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = clean_customer(df)
assert result["customer_clean"].tolist() == ["Alice Smith", "Bob Doe", "Carol", "Diana", "Eve Woods", "Frank"]
print("ok")


## Task 3 — Vectorize multi-branch logic with `np.select`

Replace this slow function with a vectorized version using `np.select`:

```python
# Slow original:
def classify(amount):
    if amount < 50:    return "small"
    if amount < 200:   return "medium"
    return "large"
df["size"] = df["amount"].apply(classify)
```

```python
def add_size(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

`np.select([cond1, cond2], [val1, val2], default=...)` evaluates the conditions in order and picks the first match. Don't forget the default for values that match none.

Expected `size`: `["medium", "small", "large", "medium", "large", "medium"]`.

In [ ]:
import pandas as pd
import numpy as np

def add_size(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = add_size(df)
assert result["size"].tolist() == ["medium", "small", "large", "medium", "large", "medium"]
print("ok")


## Task 4 — Map a column with `.map`

Replace single-letter region codes with full names using `Series.map(dict)`:

```
N → North
S → South
E → East
W → West
```

Return a new DataFrame with a `region_full` column. If a code isn't in the mapping, the value should be `"unknown"` (not `NaN`).

```python
def expand_region(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Hint: `.map(dict)` returns NaN for unmapped values. `.fillna("unknown")` covers that. Avoid `.apply` here too.

In [ ]:
import pandas as pd

def expand_region(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = expand_region(df)
assert result["region_full"].tolist() == ["North", "South", "North", "East", "South", "East"]

# Verify the fallback path works
weird = pd.DataFrame({"region": ["X", "N"]})
assert expand_region(weird)["region_full"].tolist() == ["unknown", "North"]
print("ok")
